# 52 — Thesis figures, Part III (Dominant design)

This notebook draws every figure and table of this part of the thesis and writes them, as PDF (for the thesis), PNG and CSV, to
`figures/Part_III_dominant_design/<group>/<role>/`. Run it top to bottom; every cell shows exactly the file that was written.

| group | role | file | kind | what it shows |
|---|---|---|---|---|
| III1. Architecture shares | main | `q6_architecture_shares_by_cohort` | figure | Architecture share per priority cohort |
| III1. Architecture shares | main | `q6_largest_architectures_over_time` | figure | Five largest architectures over time |
| III1. Architecture shares | main | `q6_share_trend_spearman` | table | Spearman rho of each architecture's share on priority year |
| III2 and III3. Dominant design | main | `q7_condition1_family` | figure | Condition 1 on F per priority year |
| III2 and III3. Dominant design | main | `q7_condition2_balance` | figure | Condition 2: ²D against its band |
| III2 and III3. Dominant design | main | `q7_condition3_cohesion` | figure | Condition 3: ΔQ against its band |
| III2 and III3. Dominant design | main | `q7_family_conditions` | figure | The three conditions on F (III.3) |
| III2 and III3. Dominant design | appendix | `q7_verdict_table` | table | The dominant-design verdict, levels A, A·W·T, A·U and F |
| III2 and III3. Dominant design | appendix | `q7_condition3_weights` | table | Weights of the form distance |
| III2 and III3. Dominant design | appendix | `q7_three_year_check` | table | Condition 1 with three consecutive years |
| III2 and III3. Dominant design | appendix | `q7_granted_live_condition1` | table | F's share on granted and live patents alone |
| III2 and III3. Dominant design | appendix | `q7_granted_live_condition2` | table | ²D of F on granted and live patents alone |
| III4. The directory's five categories | main | `q7_iii4_directory_categories` | figure | Conditions 1 and 2 on the five directory categories |
| III4. The directory's five categories | appendix | `q7_iii4_directory_fold` | table | Architectures folded to the directory categories |

**Roles:** *main* goes in the chapter, *auxiliary* backs a statement of the chapter, *appendix* goes to the appendix.

## Where the numbers come from

- **The patent dataset** (627 designs): the author's characterisation of the patent drawings, read through `viz/thesis_q6.py`, `thesis_q7.py` and `viz/analysis_iii.py`. This notebook only draws.
- **Legal status** (PatSeer, snapshot 2026-06-08): *actively protected* = at least one linked patent in force; *granted and live* = at least one granted patent in force.
- **Directory categories**: `analysis.class_fold` of `labelling_analysis.yaml` (SLC, SRW → LC; TR, TW, CVT, TB, PTC, DS → VT; MR → WM; RC → ER; HB, PFV → HB), the fold of the evtol.news comparison; Table architectureFold of Appendix D was not available to check against.
- **Permutations**: 200, seed 42; ²D rarefied to 40 designs. Disparity is not rerun for III4.
- **Terms used in the figures:** architecture, design, cohort (a window of priority years), assignee; levels of description A = architecture alone, A·W·T = architecture, number of wings and whether any propulsor tilts, A·U = architecture and propulsor-unit band; F = Lift + Cruise + CVT* family.

## Look and layout rules

Every figure is drawn under the author's thesis style, `viz/thesis.mplstyle` (style 1, chosen 2026-10-12 among five in `viz/style_options/`):
Arial-type sans (Liberation Sans here), 9 pt labels, 8 pt ticks and legends, no top or right frame, a light horizontal grid, 6.3 in wide, PDF with embedded fonts.

The rules each figure is held to, and checked against in this notebook (`viz/figcheck.py`):

1. **No text on top of other text or of a mark.** Checked automatically: the check table at the end must read 0.
2. **Anything in columns is placed at fixed positions, never padded with spaces** (for example a numbered key), so rows line up exactly.
3. **One grid per plot**: a right-hand second axis draws no grid of its own.
4. **Words, not codes**: every abbreviation in a figure is spelled out in the figure or its key.
5. **No title inside the figure**: the caption in the thesis carries it.
6. **Nothing under 6.5 pt** at print size. Checked automatically.

**Since 2026-10-20** the five figures of the dissertation (4.9 to 4.13: `q6_architecture_shares_by_cohort`, `q7_condition2_balance`, `q7_condition3_cohesion`, `q7_family_conditions`, `q7_iii4_directory_categories`) are drawn under `notebooks/thesis_style.py` (`import thesis_style as ts; ts.use()`): 6.3 in wide, Helvetica-type sans (Nimbus Sans here), nothing under 8 pt, one bottom legend, 2024–26* hollow and never joined. `ts.save` writes their pdf and 300 dpi png. 4.10 and 4.11 share one fixed panel geometry (`viz/figures_part3.PAIR_ADJUST`).

## Setup

In [ ]:
import sys, pathlib, importlib
PILLAR = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(PILLAR / "src"))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
from viz import config, thesis, figcheck
v2 = importlib.import_module('viz.figures_part3')
cfg = config.load()
ctx = thesis.context(cfg)
import thesis_style as ts; ts.use()      # after thesis.context (it applies viz.style); Figs 4.9-4.13 are drawn under this style
OUT = v2.out_root(cfg)
print(len(ctx.v), 'designs | external inputs:', cfg.analysis.get('external_inputs', 'script'), '| writing to', OUT)
CHECKS = []
FORBIDDEN = ()

def show(name):
    """Build the item once under the thesis style, write it, check its layout, and show the written file."""
    it = next(i for i in v2.ITEMS if i['name'] == name)
    obj, data = v2.save_v2_item(it, cfg, ctx)
    folder = OUT / it['group'] / it['role']
    if it['kind'] == 'figure':
        final = bool(it.get('width'))
        rep = figcheck.final_figure_report(obj, width_in=it['width'] * v2.TEXT_W_IN if final else None,
                                           forbid=FORBIDDEN, floor_pt=7.0 if final else 6.5)
        ov = figcheck.overlaps(obj)
        row = {'file': name, 'group': it['group'], 'role': it['role'], 'final size': final, **rep,
                'problems': '; '.join(f'{a} | {b}' for a, b in ov[:3])}
        CHECKS.append(row)
        plt.close('all')
        display(Image(filename=str(folder / f'{name}.png'), width=760))
        bad = {k: v for k, v in rep.items() if (isinstance(v, bool) and not v) or (not isinstance(v, (bool, float)) and v not in (0,))}
        print(('layout OK' if not bad else f'CHECK: {bad} {ov[:3]}'), '|', folder / f'{name}.pdf')
    else:
        CHECKS.append({'file': name, 'group': it['group'], 'role': it['role'], 'final size': False, 'problems': 'table'})
        display(obj)
        print(folder / f'{name}.csv')

## III1. Architecture shares

*How does each architecture's share of the designs move across cohorts, and which trends are monotonic?*

### Architecture share per priority cohort

`q6_architecture_shares_by_cohort` · main · figure · from v2 of q6_class_shares_by_window (Figure 10 (i))

In [ ]:
show('q6_architecture_shares_by_cohort')

### Five largest architectures over time

`q6_largest_architectures_over_time` · main · figure · from v2 of q6_largest_classes_over_time (Figure 10 (ii))

In [ ]:
show('q6_largest_architectures_over_time')

### Spearman rho of each architecture's share on priority year

`q6_share_trend_spearman` · main · table · from new (Analysis III item 1)

In [ ]:
show('q6_share_trend_spearman')

## III2 and III3. Dominant design

*Does one level of description, A (architecture), A·W·T, A·U or the family F, meet the three conditions: share above half, balance below the band, form below the band?*

### Condition 1 on F per priority year

`q7_condition1_family` · main · figure · from v2 of q7_condition1_family (Figure 11)

In [ ]:
show('q7_condition1_family')

### Condition 2: ²D against its band

`q7_condition2_balance` · main · figure · from v2 of q7_condition2_balance (Figure 12 (i))

In [ ]:
show('q7_condition2_balance')

### Condition 3: ΔQ against its band

`q7_condition3_cohesion` · main · figure · from v2 of q7_condition3_cohesion (Figure 12 (ii))

In [ ]:
show('q7_condition3_cohesion')

### The three conditions on F (III.3)

`q7_family_conditions` · main · figure · new (hand-over 2026-10-16, Task 3): condition 1 (share per priority year, three bases), 2 (²D) and 3 (ΔQ) of F, all designs with the band and the actively protected series in grey.

In [ ]:
show('q7_family_conditions')

### The dominant-design verdict, levels A, A·W·T, A·U and F

`q7_verdict_table` · appendix · table · from v2 of q7_verdict_table (Table 8)

In [ ]:
show('q7_verdict_table')

In [ ]:
show('q7_verdict_detail')

In [ ]:
show('q7_condition3_family')

### Weights of the form distance

`q7_condition3_weights` · appendix · table · from new (Analysis III item 2)

In [ ]:
show('q7_condition3_weights')

### Condition 1 with three consecutive years

`q7_three_year_check` · appendix · table · from new (Analysis III item 4)

In [ ]:
show('q7_three_year_check')

### F's share on granted and live patents alone

`q7_granted_live_condition1` · appendix · table · from new (Analysis III item 5)

In [ ]:
show('q7_granted_live_condition1')

### ²D of F on granted and live patents alone

`q7_granted_live_condition2` · appendix · table · from new (Analysis III item 5)

In [ ]:
show('q7_granted_live_condition2')

## III4. The directory's five categories

*Do the conditions hold when the twelve architectures are folded to the five categories of the public directory?*

### Conditions 1 and 2 on the five directory categories

`q7_iii4_directory_categories` · main · figure · from new (Analysis III item 6)

In [ ]:
show('q7_iii4_directory_categories')

### Architectures folded to the directory categories

`q7_iii4_directory_fold` · appendix · table · from analysis.class_fold

In [ ]:
show('q7_iii4_directory_fold')

## Layout check, all figures

Every count must read 0 and every flag True: text over text, text under the font floor (7 pt for the figures printed in the dissertation, 6.5 pt for the others), text outside the canvas, forbidden words, the final width, and the height of at most 55 % of the width.

In [ ]:
chk = pd.DataFrame(CHECKS)
display(chk)
figs = chk[chk['problems'] != 'table']
num = [c for c in figs.columns if c in ('text over text', 'under 7 pt', 'under 6 pt', 'outside canvas', 'forbidden words')]
flags = [c for c in ('width_ok', 'height_ok') if c in figs.columns]
ok = all((figs[c].fillna(0) == 0).all() for c in num) and all(figs[c].dropna().astype(bool).all() for c in flags)
print('all figures pass' if ok else 'SOME FIGURES NEED A FIX (see the table)')

## Hand-over to the dissertation (latex)

The four figures that stay in the text and the two that leave it, redrawn at their final width in the 16 cm text block (1.0, 0.9, 0.85, 1.0; height at most 55 % of the width), with the data behind every figure and table, `NUMBERS_III.md` and the manifest lines. Nothing is recomputed differently; the builders are those above.

In [ ]:
from viz import thesis_latex_part3 as hand
chk = hand.export(cfg, ctx)
display(chk)
print('all pass' if (chk['text over text'] == 0).all() and (chk['under 7 pt'] == 0).all() and (chk['height / width'] <= 0.55).all() else 'FIX NEEDED')
print(hand.root(cfg))

## README and file list

Rewrite the folder's README.md and list every file written.

In [ ]:
import subprocess
print(subprocess.run([sys.executable, str(PILLAR / 'notebooks' / '_build' / 'write_readme_52.py')], capture_output=True, text=True).stdout)
files = sorted(p.relative_to(OUT) for p in OUT.rglob('*') if p.is_file())
print(len(files), 'files'); files

## Hand-over 2026-10-16 (Tasks 1 and 3)

Task 1: condition 3 rerun under the 15-dimension weighting `w15` (`frames.FORM15`, `frames.form_distance`; `w34` kept in the code for the record; same seed 42, 200 permutations, rarefaction and bases). The cells above wrote `q7_condition3_cohesion`, `q7_condition3_family`, `q7_condition3_weights`, `q7_verdict_table`, `q7_verdict_detail` and `NUMBERS_III.md`.
Task 3: F out of `q7_condition2_balance` and `q7_condition3_cohesion` (three panels A, A·W·T, A·U) and into the new `q7_family_conditions`; `q7_condition1_family` stays written but leaves the text.

This cell writes the other copies the hand-over asks for: png, csv and json beside the latex pdfs (drawn at the final width), the notebook-size (v2) copies under `latex/Figures/figures/...`, the Q7 items through the builder and writer of notebook 46 (`thesis_figures/Q7_dominant_design/results/`), and every file plus the changed scripts in `handoff_2026-10-16/` at the repository root.

In [ ]:
from viz import handoff as ho, thesis_latex_part3 as hand, thesis_q7 as q7
R = hand.root(cfg)
PART = 'Part_III_dominant_design'
written = []
TASK_FIGS = {'task1': ['q7_condition3_cohesion'], 'task3': ['q7_condition2_balance', 'q7_condition3_cohesion', 'q7_family_conditions']}
TASK_TABLES = {'task1': ['q7_condition3_family', 'q7_condition3_weights', 'q7_verdict_table', 'q7_verdict_detail']}
for task, names in TASK_FIGS.items():
    for name in names:
        it = next(i for i in v2.ITEMS if i['name'] == name)
        written += ho.figure_copies(task, name, it, PART, OUT / it['group'] / it['role'], R, lambda it=it, name=name: hand.draw(it, ctx, hand.FIGS[name][0])[:2])
        written += ho.q_item_copies(task, name, q7.ITEMS, ctx, cfg, suffix='q7results')
for task, names in TASK_TABLES.items():
    for name in names:
        it = next(i for i in v2.ITEMS if i['name'] == name)
        written += ho.table_copies(task, name, it, PART, OUT / it['group'] / it['role'], R)
written += ho.plain_copies('task1', [R / 'Data_for_writing' / 'NUMBERS_III.md'])
ho.copy_scripts('task1', ['lib/distance.py', 'viz/frames.py', 'viz/analysis_iii.py', 'viz/thesis_q7.py', 'viz/thesis_text_q7.py', 'viz/thesis_latex_part3.py',
                          'viz/figures_part3.py', 'viz/handoff.py', 'notebooks/23_dominant_design.ipynb'])
ho.copy_scripts('task3', ['viz/thesis_q7.py', 'viz/plots.py', 'viz/figures_part3.py', 'viz/thesis_latex_part3.py', 'viz/handoff.py',
                          'notebooks/23_dominant_design.ipynb'])
print('\n'.join(map(str, written)))
print(ho.listing('task1') + ho.listing('task3'))

In [ ]:
# 2026-10-31 re-export (Analysis III review): Fig 4.11, panel (c) y label "ΔQ of the cohort" (condition 3 is computed on the
# whole cohort). Written to figures by show(), then copied to latex (as README 2026-10-29). Nothing else changes.
import shutil
from viz import thesis_latex_part3 as hand
show('q7_family_conditions')
it = next(i for i in v2.ITEMS if i['name'] == 'q7_family_conditions')
dst = hand.root(cfg) / 'Figures' / 'Part_III_dominant_design' / it['group'] / it['role']
for ext in ('pdf', 'png'):
    shutil.copy2(OUT / it['group'] / it['role'] / f'q7_family_conditions.{ext}', dst / f'q7_family_conditions.{ext}')
    print(dst / f'q7_family_conditions.{ext}')